# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tracy030115/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

K-Means clustering

It doesn't need me to decide the baseline boundaries in advance. It looks at all the scaled metrics together, impressions, clicks, CTR, position, engagement rate, days since update, and groups pages that are actually similar across all of them at once, instead of pages that happen to fall on the same side of a few cutoffs I chose one at a time. Real archetypes are combinations across several metrics, not one threshold, so this fits the lane better than anything rule based could.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split by client

Several of the flagged archetypes, engagement_problem_pages, stale_visible_pages, kept clustering around a single client instead of spreading across many. If I split randomly by row, pages from the same client end up in both the train set and the test set. Any client level pattern, like one client just updating content less often, or one client having a GA4 tracking quirk, would then leak into both splits. The model would look like it generalizes well, but it would really just be learning that specific client's habits and getting rewarded for recognizing them again in the other split. Splitting by client_hash_id, so all of one client's pages stay together in either train or test, is a way to actually test whether the archetypes generalize to a client the method has never seen.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

page_level = con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    fact_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(ga4_sessions) AS total_sessions,
            SUM(ga4_engaged_sessions) AS total_engaged_sessions,
            BOOL_OR(gsc_data_available) AS gsc_data_available_any,
            BOOL_OR(ga4_data_available) AS ga4_data_available_any
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    content_meta AS (
        SELECT
            content_hash_id,
            keyword_hash_id,
            content_updated_date,
            is_published,
            is_deleted
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published = true AND is_deleted = false
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        cm.keyword_hash_id,
        f.total_impressions,
        f.total_clicks,
        f.avg_position,
        f.total_sessions,
        f.total_engaged_sessions,
        f.gsc_data_available_any,
        f.ga4_data_available_any,
        DATE_DIFF('day', cm.content_updated_date, ref.max_date) AS days_since_last_update,
        CASE WHEN f.total_impressions > 0 THEN f.total_clicks * 1.0 / f.total_impressions ELSE NULL END AS ctr,
        CASE WHEN f.total_sessions > 0 THEN f.total_engaged_sessions * 1.0 / f.total_sessions ELSE NULL END AS engagement_rate
    FROM fact_agg f
    JOIN content_meta cm ON f.content_hash_id = cm.content_hash_id
    CROSS JOIN ref
""").df()

print(page_level.shape)
page_level.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(394928, 13)


,client_hash_id,content_hash_id,keyword_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_engaged_sessions,gsc_data_available_any,ga4_data_available_any,days_since_last_update,ctr,engagement_rate
0,client_f623b01661d4bfe4,content_575a1ed6e670a9a3,keyword_e58d14f168cd68dc,2.0,0.0,48.00,0.0,0.0,True,False,41,0.0,NaN
1,client_f623b01661d4bfe4,content_22290552590d7529,keyword_c1224858d58b5062,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
2,client_f623b01661d4bfe4,content_bc7ee2e19eb9ce58,keyword_339d84789c583edf,6.0,0.0,20.80,0.0,0.0,True,False,41,0.0,NaN
3,client_f623b01661d4bfe4,content_5a8d5e9988c5cab3,keyword_ec9c29f64307841f,3.0,0.0,0.50,0.0,0.0,True,False,41,0.0,NaN
4,client_f623b01661d4bfe4,content_ef950fcfe31b8458,keyword_922a0faa27b45a82,5.0,0.0,7.25,0.0,0.0,True,False,41,0.0,NaN


In [7]:
def position_bucket(pos):
    if pos is None or pd.isna(pos):
        return None
    if pos <= 3:
        return "1_pos_1-3"
    elif pos <= 10:
        return "2_pos_4-10"
    elif pos <= 20:
        return "3_pos_11-20"
    else:
        return "4_pos_21plus"

expected_ctr_by_bucket = {
    "1_pos_1-3": 0.027805,
    "2_pos_4-10": 0.004229,
    "3_pos_11-20": 0.004184,
    "4_pos_21plus": 0.002788,
}

page_level["position_bucket"] = page_level["avg_position"].apply(position_bucket)
page_level["expected_ctr"] = page_level["position_bucket"].map(expected_ctr_by_bucket)
page_level["ctr_gap"] = page_level["expected_ctr"] - page_level["ctr"]

In [8]:
keyword_counts = (
    page_level[page_level["total_impressions"] > 0]
    .groupby(["client_hash_id", "keyword_hash_id"])["content_hash_id"]
    .nunique()
    .reset_index(name="pages_ranking_for_keyword")
)
page_level = page_level.merge(keyword_counts, on=["client_hash_id", "keyword_hash_id"], how="left")
page_level["cannibalization_risk"] = page_level["pages_ranking_for_keyword"].fillna(0) > 1

In [9]:
MIN_IMPRESSIONS = 10
STALE_THRESHOLD_DAYS = 180
HIGH_VOLUME_IMPRESSIONS = page_level["total_impressions"].quantile(0.75)

def assign_archetype(row):
    gsc_available = bool(row["gsc_data_available_any"]) if pd.notna(row["gsc_data_available_any"]) else False
    if (not gsc_available) or row["total_impressions"] < MIN_IMPRESSIONS:
        return "INSUFFICIENT_DATA"

    # negative days_since_last_update is a reporting lag artifact, not real freshness
    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] < 0:
        return "INSUFFICIENT_DATA"

    is_good_position = pd.notna(row["avg_position"]) and row["avg_position"] <= 10
    is_high_volume = row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS
    is_low_ctr = pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0
    is_stale = pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > STALE_THRESHOLD_DAYS
    has_low_engagement = pd.notna(row["engagement_rate"]) and row["engagement_rate"] < 0.3
    is_no_demand = row["total_impressions"] < MIN_IMPRESSIONS * 3
    ga4_available = bool(row["ga4_data_available_any"]) if pd.notna(row["ga4_data_available_any"]) else False
    is_cannibalization = bool(row["cannibalization_risk"]) if pd.notna(row["cannibalization_risk"]) else False

    if is_cannibalization:
        return "cannibalization_risk"
    if is_good_position and is_high_volume and not is_low_ctr and not is_stale:
        return "champions"
    if is_good_position and is_high_volume and is_stale:
        return "stale_visible_pages"
    if ga4_available and is_good_position and is_high_volume and has_low_engagement:
        return "engagement_problem_pages"
    if is_good_position and not is_high_volume and not is_low_ctr:
        return "hidden_gems"
    if not is_good_position and pd.notna(row["ctr_gap"]) and row["ctr_gap"] < 0 and row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS * 0.25:
        return "rising_stars"
    if is_no_demand:
        return "weak_no_demand_pages"

    return "monitor"

page_level["archetype"] = page_level.apply(assign_archetype, axis=1)
page_level["archetype"].value_counts()

,count
archetype,
INSUFFICIENT_DATA,272549
monitor,56390
rising_stars,19210
weak_no_demand_pages,17426
champions,14300
engagement_problem_pages,12752
hidden_gems,2272
stale_visible_pages,27
cannibalization_risk,2


In [17]:
# 1. Assign clients to train/test groups (not by date, just to keep each client's rows together)
np.random.seed(42)
all_clients = page_level["client_hash_id"].unique()
np.random.shuffle(all_clients)

n_test_clients = max(1, int(len(all_clients) * 0.2))
test_clients = set(all_clients[:n_test_clients])
train_clients = set(all_clients[n_test_clients:])

print(f"train clients: {len(train_clients)}, test clients: {len(test_clients)}")

train clients: 52, test clients: 12


In [18]:
# 2. Separately, pick a date cutoff using the full report_date range (this is about
# time awareness within the fact table, not about client last_date)
time_cutoff = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

min_date = pd.to_datetime(time_cutoff["min_date"][0])
max_date = pd.to_datetime(time_cutoff["max_date"][0])
print(f"data spans {min_date.date()} to {max_date.date()}")

data spans 2026-06-01 to 2026-06-30


In [19]:
con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS n_distinct_dates
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,min_date,max_date,n_distinct_dates
0,2026-06-01,2026-06-30,30


In [20]:
page_level["split"] = np.where(
    page_level["client_hash_id"].isin(train_clients), "train", "test"
)
page_level["split"].value_counts()

,count
split,
train,357842
test,37086


In [33]:
feature_cols = ["total_impressions", "total_clicks", "ctr", "avg_position", "days_since_last_update"]

model_data = page_level[page_level["archetype"] != "INSUFFICIENT_DATA"].copy()
model_data = model_data.dropna(subset=feature_cols)
print(f"model_data rows: {len(model_data)}")

train = model_data[model_data["client_hash_id"].isin(train_clients)].copy()
test = model_data[model_data["client_hash_id"].isin(test_clients)].copy()
print(f"train rows: {len(train)}, test rows: {len(test)}")

model_data rows: 122379
train rows: 102439, test rows: 19940


In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test = scaler.transform(test[feature_cols])

k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
train_clusters = kmeans.fit_predict(X_train)
test_clusters = kmeans.predict(X_test)

train["kmeans_cluster"] = train_clusters
test["kmeans_cluster"] = test_clusters

train_silhouette = silhouette_score(X_train, train_clusters)
test_silhouette = silhouette_score(X_test, test_clusters)
print(f"train silhouette: {train_silhouette:.3f}")
print(f"test silhouette: {test_silhouette:.3f}")

train silhouette: 0.485
test silhouette: 0.526


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Where the model is wrong: K-Means at k=7 does not produce 7 behaviorally distinct archetypes. One cluster, holding roughly 60% of all clustered pages across two separate runs of different sizes, absorbs the majority of every rule based archetype except stale_visible_pages. Seven of eight archetype labels have their dominant K-Means cluster share below 0.77, several below 0.55, meaning pages the rule based system considers meaningfully different, a champion versus a page under monitor versus a weak no demand page, are landing in the same K-Means cluster far more often than not.

In [40]:
# Step 3: Where K-Means and the rule based archetype disagree the most

# cross tab of rule based archetype vs kmeans cluster assignment
overlap = pd.crosstab(train["archetype"], train["kmeans_cluster"])

# for each rule based archetype, find which single kmeans cluster holds
# the largest share of its rows, and what that share actually is
dominant_cluster = overlap.idxmax(axis=1)
dominant_share = overlap.max(axis=1) / overlap.sum(axis=1)

agreement_summary = pd.DataFrame({
    "dominant_kmeans_cluster": dominant_cluster,
    "share_in_dominant_cluster": dominant_share.round(3),
    "total_rows": overlap.sum(axis=1)
}).sort_values("share_in_dominant_cluster")

agreement_summary

,dominant_kmeans_cluster,share_in_dominant_cluster,total_rows
archetype,,,
hidden_gems,1,0.476,1961
cannibalization_risk,1,0.500,2
weak_no_demand_pages,1,0.516,14629
monitor,1,0.533,47379
rising_stars,1,0.685,16121
engagement_problem_pages,1,0.718,10377
champions,1,0.766,11947
stale_visible_pages,0,0.957,23


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.